# Метрики пайплайна и бейзлайнов (финальная сводка)

Запускать **после** того как:
1. Прогнан `tfidf-bm25-pool-extend.ipynb` (расширен pool).
2. В `pipeline-50k-major_metrics-annotation.ipynb` доразмечены новые пары
   (`ground_truth_pool.json` покрывает весь расширенный pool).

Здесь на одном и том же финальном пуле считаются три метрики (Recall@10,
MRR, nDCG@10) для **четырёх** методов:
- **TF-IDF** — классический лексический поиск;
- **BM25** — улучшенная лексическая модель Okapi BM25;
- **retrieve-only (E5 fine-tuned)** — только семантический поиск;
- **retrieve+rerank (E5 fine-tuned + DiTy cross-encoder fine-tuned)** — полный пайплайн.


In [1]:
# Поднимаемся к корню thesis/
import os
from pathlib import Path
_p = Path.cwd()
if _p.name == 'baselines':
    os.chdir(_p.parent.parent)
print('CWD:', os.getcwd())


CWD: c:\Users\Admin\Documents\диплом\thesis


In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import math
import time
import re
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display

# sentence-transformers 5.x совместимость
import transformers
from transformers.modeling_utils import PreTrainedModel
transformers.PreTrainedModel = PreTrainedModel

from sentence_transformers import SentenceTransformer, CrossEncoder
import lancedb

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from rank_bm25 import BM25Okapi

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {DEVICE}')


DEVICE: cpu


In [3]:
# ==================== КОНФИГУРАЦИЯ ====================

BI_ENCODER_PATH    = 'models/bi-encoder-e5-finetuned'
BI_DOC_PREFIX      = 'passage: '
BI_QUERY_PREFIX    = 'query: '

CROSS_ENCODER_PATH = 'models/final/cross-encoder'

LANCEDB_PATH       = './lancedb_store'
POOL_TABLE         = 'e5-base-fine-tuned-50k'

GT_PAIRS_JSON        = 'ground_truth_pairs.json'
POOL_CANDIDATES_JSON = 'benchmark/pipeline/pool_candidates.json'
GT_POOL_JSON         = 'benchmark/pipeline/ground_truth_pool.json'

DROP_QUERY_INDEXES  = {6, 14}

TOP_K_BI            = 10    # финальная глубина для retrieve-only
TOP_K_BI_FOR_RERANK = 100
K_METRIC            = 10

TOKEN_RE = re.compile(r'[А-Яа-яЁёA-Za-z0-9]+')

def simple_tokenizer(text):
    return TOKEN_RE.findall(text.lower())


## 1. Запросы и pool

In [4]:
with open(GT_PAIRS_JSON, encoding='utf-8') as f:
    all_pairs = json.load(f)
queries = []
for i, pair in enumerate(all_pairs, start=1):
    if i in DROP_QUERY_INDEXES:
        continue
    queries.append({'query_idx': i, 'imt_name': pair['imt_name'], 'description': pair['description']})
print(f'Запросов: {len(queries)}')

with open(POOL_CANDIDATES_JSON, encoding='utf-8') as f:
    pool = json.load(f)
pool_by_q = {p['query_idx']: p for p in pool}
total_pairs = sum(len(p['candidates']) for p in pool)
print(f'Pool: {len(pool)} запросов, {total_pairs} пар')

with open(GT_POOL_JSON, encoding='utf-8') as f:
    annotations = json.load(f)
coverage = len(annotations) / total_pairs
print(f'Разметка: {len(annotations)}/{total_pairs} ({coverage:.1%})')
assert coverage >= 0.99, f'Pool размечен не полностью — доразметьте в annotation-ноутбуке'


Запросов: 17
Pool: 17 запросов, 814 пар
Разметка: 814/814 (100.0%)


## 2. Загрузка моделей и индексов

In [5]:
# 2.1. База постов
db = lancedb.connect(LANCEDB_PATH)
tbl = db.open_table(POOL_TABLE)
df = tbl.to_pandas()
print(f'Постов: {len(df):,}')

# 2.2. Bi-encoder (E5)
bi_encoder = SentenceTransformer(BI_ENCODER_PATH, device=DEVICE)
print('bi-encoder загружен')

# 2.3. Cross-encoder (DiTy)
cross_encoder = CrossEncoder(CROSS_ENCODER_PATH, device=DEVICE)
print('cross-encoder загружен')


You are trying to use a model that was created with Sentence Transformers version 5.3.0, but you're currently using version 5.2.3. This might cause unexpected behavior or errors. In that case, try to update to the latest version.


Постов: 50,000


The tokenizer you are loading from 'models/final/cross-encoder' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


bi-encoder загружен
cross-encoder загружен


In [6]:
# 2.4. TF-IDF индекс
t0 = time.time()
tfidf_vec = TfidfVectorizer(
    tokenizer=simple_tokenizer, lowercase=False,
    ngram_range=(1, 2), min_df=2, max_df=0.95, norm='l2',
)
tfidf_matrix = tfidf_vec.fit_transform(df['text'].tolist())
print(f'TF-IDF: {tfidf_matrix.shape}, {time.time()-t0:.1f}с')

# 2.5. BM25 индекс
t0 = time.time()
tokenized_corpus = [simple_tokenizer(t) for t in df['text'].tolist()]
bm25 = BM25Okapi(tokenized_corpus)
print(f'BM25 построен за {time.time()-t0:.1f}с')


TF-IDF: (50000, 493042), 4.4с
BM25 построен за 1.5с


## 3. Функции retrieval / rerank

In [7]:
def retrieve_bi(query, k):
    q = BI_QUERY_PREFIX + query
    q_emb = bi_encoder.encode([q], normalize_embeddings=True, show_progress_bar=False)
    results = tbl.search(q_emb[0]).limit(k).to_list()
    return [{'text': r['text'], 'channel': r['channel'], 'category': r.get('category', '')} for r in results]


def retrieve_tfidf(query, k):
    q_vec = tfidf_vec.transform([query])
    sims = linear_kernel(q_vec, tfidf_matrix).flatten()
    top = np.argpartition(-sims, kth=k)[:k]
    top = top[np.argsort(-sims[top])]
    return [{'text': df.iloc[int(i)]['text'], 'channel': df.iloc[int(i)]['channel'], 'category': df.iloc[int(i)].get('category', '')} for i in top]


def retrieve_bm25(query, k):
    scores = bm25.get_scores(simple_tokenizer(query))
    top = np.argpartition(-scores, kth=k)[:k]
    top = top[np.argsort(-scores[top])]
    return [{'text': df.iloc[int(i)]['text'], 'channel': df.iloc[int(i)]['channel'], 'category': df.iloc[int(i)].get('category', '')} for i in top]


def rerank_cross(query, candidates, k):
    pairs = [(query, c['text']) for c in candidates]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)
    order = np.argsort(-scores)
    return [candidates[int(i)] for i in order[:k]]


## 4. Функции метрик

In [8]:
def relevance(query_idx, post_text):
    # ищем пост в пуле по тексту, возвращаем метку из annotations (или 0)
    p = pool_by_q[query_idx]
    for ci, c in enumerate(p['candidates']):
        if c['text'].strip() == post_text.strip():
            return int(annotations.get(f'{query_idx}:{ci}', 0))
    return 0


def all_relevances_for(query_idx):
    p = pool_by_q[query_idx]
    rels = []
    for ci, c in enumerate(p['candidates']):
        r = int(annotations.get(f'{query_idx}:{ci}', 0))
        if r >= 1:
            rels.append(r)
    return rels


def recall_at_k(rels_in_top_k, n_relevant):
    found = sum(1 for r in rels_in_top_k if r >= 1)
    return found / n_relevant if n_relevant else None


def reciprocal_rank(rels_in_top_k):
    for i, r in enumerate(rels_in_top_k, 1):
        if r >= 1:
            return 1.0 / i
    return 0.0


def ndcg_at_k(rels_in_top_k, all_rels, k):
    dcg = sum((2**rel - 1) / math.log2(i + 2) for i, rel in enumerate(rels_in_top_k[:k]))
    ideal = sorted(all_rels, reverse=True)[:k]
    idcg = sum((2**rel - 1) / math.log2(i + 2) for i, rel in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_run(runs, k=K_METRIC):
    rs, mr, nd = [], [], []
    for r in runs:
        qi = r['query_idx']
        top_texts = r['top_k_texts']
        all_rels = all_relevances_for(qi)
        if not all_rels:
            continue
        rels_in_top = [relevance(qi, t) for t in top_texts]
        rs.append(recall_at_k(rels_in_top, len(all_rels)))
        mr.append(reciprocal_rank(rels_in_top))
        nd.append(ndcg_at_k(rels_in_top, all_rels, k))
    return {
        'Recall@10': float(np.mean(rs)),
        'MRR':       float(np.mean(mr)),
        'nDCG@10':   float(np.mean(nd)),
        'n_queries': len(rs),
    }


## 5. Прогон всех четырёх методов

In [9]:
runs = {'TF-IDF': [], 'BM25': [], 'retrieve-only (E5)': [], 'retrieve+rerank (E5 + DiTy)': []}

for q in queries:
    qi = q['query_idx']
    d  = q['description']

    # TF-IDF top-10
    top_tfidf = retrieve_tfidf(d, K_METRIC)
    runs['TF-IDF'].append({'query_idx': qi, 'top_k_texts': [c['text'] for c in top_tfidf]})

    # BM25 top-10
    top_bm25 = retrieve_bm25(d, K_METRIC)
    runs['BM25'].append({'query_idx': qi, 'top_k_texts': [c['text'] for c in top_bm25]})

    # retrieve-only E5 top-10
    top_bi = retrieve_bi(d, K_METRIC)
    runs['retrieve-only (E5)'].append({'query_idx': qi, 'top_k_texts': [c['text'] for c in top_bi]})

    # retrieve+rerank E5+DiTy: bi top-100 → cross top-10
    cand100 = retrieve_bi(d, TOP_K_BI_FOR_RERANK)
    top_rr  = rerank_cross(d, cand100, K_METRIC)
    runs['retrieve+rerank (E5 + DiTy)'].append({'query_idx': qi, 'top_k_texts': [c['text'] for c in top_rr]})

print(f'Прогнаны все 4 метода × {len(queries)} запросов')


Прогнаны все 4 метода × 17 запросов


In [10]:
metrics = {name: evaluate_run(rs) for name, rs in runs.items()}

order = ['TF-IDF', 'BM25', 'retrieve-only (E5)', 'retrieve+rerank (E5 + DiTy)']
rows = []
for name in order:
    m = metrics[name]
    rows.append({'Метод': name, 'Recall@10': round(m['Recall@10'], 4),
                 'MRR': round(m['MRR'], 4), 'nDCG@10': round(m['nDCG@10'], 4)})

df_metrics = pd.DataFrame(rows)
print(f'Финальная таблица на pool-GT ({len(queries)} запросов, всего {total_pairs} размеченных пар)')
display(df_metrics.style.background_gradient(cmap='YlGn', subset=['Recall@10', 'MRR', 'nDCG@10']))


Финальная таблица на pool-GT (17 запросов, всего 814 размеченных пар)


,Метод,Recall@10,MRR,nDCG@10
0,TF-IDF,0.275500,0.791200,0.419800
1,BM25,0.268300,0.761400,0.397400
2,retrieve-only (E5),0.328500,0.746100,0.504500
3,retrieve+rerank (E5 + DiTy),0.399900,0.854900,0.591600


In [11]:
# Дельты между соседними уровнями
print('\nДельты (каждая строка — прирост относительно предыдущего метода):')
for i in range(1, len(order)):
    prev, cur = order[i-1], order[i]
    print(f'  {cur} − {prev}:')
    for k in ['Recall@10', 'MRR', 'nDCG@10']:
        d = metrics[cur][k] - metrics[prev][k]
        arrow = '↑' if d > 0 else ('↓' if d < 0 else '=')
        print(f'    Δ {k}: {d:+.4f}  {arrow}')



Дельты (каждая строка — прирост относительно предыдущего метода):
  BM25 − TF-IDF:
    Δ Recall@10: -0.0072  ↓
    Δ MRR: -0.0297  ↓
    Δ nDCG@10: -0.0224  ↓
  retrieve-only (E5) − BM25:
    Δ Recall@10: +0.0602  ↑
    Δ MRR: -0.0154  ↓
    Δ nDCG@10: +0.1071  ↑
  retrieve+rerank (E5 + DiTy) − retrieve-only (E5):
    Δ Recall@10: +0.0714  ↑
    Δ MRR: +0.1088  ↑
    Δ nDCG@10: +0.0870  ↑


## 6. Детализация по запросам (опционально)

Можно посмотреть, на каких запросах какой метод лучше/хуже —
полезно для качественного анализа в тексте диплома.


In [12]:
per_q = []
for q in queries:
    qi = q['query_idx']
    all_rels = all_relevances_for(qi)
    if not all_rels:
        continue
    row = {'q': qi, 'товар': q['imt_name'][:40], '|rel|': len(all_rels)}
    for name in order:
        r = next(rr for rr in runs[name] if rr['query_idx'] == qi)
        rels_in_top = [relevance(qi, t) for t in r['top_k_texts']]
        row[name] = round(ndcg_at_k(rels_in_top, all_rels, K_METRIC), 3)
    per_q.append(row)
pd.DataFrame(per_q)


,q,товар,|rel|,TF-IDF,BM25,retrieve-only (E5),retrieve+rerank (E5 + DiTy)
0,1,Затирка для плитки готовая - белая,12,0.506,0.497,0.553,0.640
1,2,Самоклеящиеся панели для стен на кухню 6,18,0.058,0.050,0.724,0.707
2,3,Развивашки 2-3-4 года/пиши стирай тетрад,18,0.395,0.212,0.738,0.651
3,4,"Детская мозаика (5 цветов, 40 элементов)",17,0.171,0.211,0.224,0.381
4,5,Жиросжигатель для похудения женщинам 60,15,0.356,0.652,0.506,0.000
5,7,Накидка на сиденье DongFeng Fengshen Yix,9,0.275,0.177,0.000,0.154
6,8,Кроссовер Monjaro,20,0.614,0.623,0.574,0.691
7,9,Матрас надувной двуспальный 203х152см с,9,0.276,0.181,0.276,0.359
8,10,Гуд Найт Мягкое фито снотворное для сна,22,0.677,0.501,0.642,0.372
9,11,Клавиатура игровая с подсветкой\n,14,0.225,0.000,0.021,0.677
